# TP 3 - Agent de voyage avec outils

Ce notebook construit un agent de planification de voyage autonome.
L'agent dispose d'un ensemble d'**outils Python** (géocodage, météo, recherche de lieux, RAG, web) et décide seul quels outils appeler et dans quel ordre pour répondre à une requête.

### 0.1. Objectif
- **TP 3_1** : Coder les outils et l'infrastructure d'exécution, puis construire un agent PydanticAI qui les orchestre
- **TP 3_2** : Migrer les outils vers une architecture MCP (serveurs locaux et distants)

### 0.2. Documentation générale

[PydanticAI](https://ai.pydantic.dev/)

[Google GenAI Python SDK](https://googleapis.github.io/python-genai/)

[Tavily Python SDK](https://docs.tavily.com/sdk/python/reference)

[Google Maps Python Client](https://github.com/googlemaps/google-maps-services-python)

[Open-Meteo API](https://open-meteo.com/en/docs)

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel, GoogleModelSettings
from pydantic_ai.providers.google import GoogleProvider

from shared.agent_tools import (
    tool_get_current_date,
    tool_geocode_location,
    tool_get_weather,
    tool_search_nearby,
    tool_retrieve_docs,
    web_search,
    web_extract,
)
from shared.agent_utils import LoggingAgent
from shared.config import ROOT_DIR, project_settings

LOG_DIR = ROOT_DIR / "TP3_travel_planner_Agent" / "logs"

### 0.3. Récapitulatif des fonctions utilisées dans ce notebook

**Fonctions et classes à utiliser**

- `EventStreamHandler` : classe qui formate les appels outils et leurs résultats en temps réel
- `LoggingAgent` : agent PydanticAI étendu avec `run_with_logging` pour l'exécution tracée et le log JSON complet

--> Disponibles dans `shared/agent_utils.py`

- `project_settings` : objet qui centralise la configuration (clés API, noms de modèles)

--> Disponible dans `shared/config.py`

- `Place` : format de lieu normalisé (nom + coordonnées GPS)
- `tool_get_current_date` : retourne la date courante pour ancrer les dates relatives
- `tool_geocode_location` : convertit un lieu texte en coordonnées GPS (Google Maps Geocoding API)
- `tool_get_weather` : retourne une prévision météo sur une plage de dates (Open-Meteo API)
- `tool_search_nearby` : cherche des lieux proches autour de coordonnées (Google Maps Places API)
- `tool_retrieve_docs` : récupère les passages RAG internes les plus pertinents
- `web_search` : lance une recherche web externe (Tavily Search API)
- `web_extract` : lit le contenu de pages web trouvées (Tavily Extract API)

--> Disponibles dans `shared/agent_tools.py`

---
## 1. Création de l'agent

### 1.1. Premiers tests de l'agent

Regardons ce que donne un appel simple à `Agent.run()`, sans outils.

In [ ]:
model = GoogleModel(
    model_name=project_settings.llm_model_name,
    provider=GoogleProvider(api_key=project_settings.google_api_generative_key),
)

agent_model_settings = GoogleModelSettings(
    temperature=project_settings.llm_temperature,
    top_p=project_settings.llm_top_p,
    max_tokens=project_settings.llm_max_output_tokens,
    google_thinking_config={"thinking_budget": project_settings.llm_thinking_budget},
)

# Question simple pour tester l'agent
test_query = "Quelle est la météo à Paris ?"

test_agent = Agent(model=model, model_settings=agent_model_settings)

test_result = await test_agent.run(test_query)
print(test_result.output)

--> Pour l'instant, notre agent fonctionne comme un simple LLM, il **ne dispose pas d'outils** pour répondre à la question.

### 1.2. Classe LoggingAgent

Pour comprendre le raisonnement de l'agent, nous voulons **tracer en temps réel** les appels aux outils et leurs résultats, tout en **sauvegardant un log complet** de la session.

`LoggingAgent` étend `Agent` et ajoute une méthode `run_with_logging()` qui fait exactement cela. Fournie dans `shared/agent_utils.py`.

---
## 2. Outils de l'agent

Tous les outils sont fournis dans `shared/agent_tools.py`, on les teste ici pour vérifier qu'ils fonctionnent correctement.

### 2.1. Données et dates

`Place` est le format de lieu normalisé partagé entre les outils. `tool_get_current_date` ancre les dates relatives dans le temps calendrier : sans lui, l'agent ne peut pas calculer "la semaine prochaine" ou "dans 3 jours". Fournis dans `shared/agent_tools.py`.

In [ ]:
# Test : outil date
tool_get_current_date()

### 2.2. Géolocalisation et météo

`tool_geocode_location` transforme un texte libre ("Rome, Italie") en coordonnées GPS via la Google Maps Geocoding API. `tool_get_weather` retourne les prévisions journalières sur une plage de dates via l'API Open-Meteo (gratuite, sans clé). Fournis dans `shared/agent_tools.py`.

In [ ]:
# Test : géolocalisation et météo
geocode_result = tool_geocode_location("Rome, Italie")
print(geocode_result)
current_date = tool_get_current_date()["current_date"]
weather_result = tool_get_weather(geocode_result["latitude"], geocode_result["longitude"], current_date, current_date)
print(weather_result)

### 2.3. Recherche de lieux et documents

`tool_search_nearby` cherche des lieux autour de coordonnées GPS via la Google Maps Places API. `tool_retrieve_docs` expose la recherche RAG construite en TP2 (`chroma_db_rag_v2`) comme un **outil de l'agent** : il récupère les passages pertinents pour une requête. Fournis dans `shared/agent_tools.py`.

In [ ]:
# Test : lieux proches et RAG
nearby_result = tool_search_nearby(41.9028, 12.4964, place_type="museum")
print(nearby_result)
rag_result = tool_retrieve_docs("Rome monuments historiques", top_k=3)
print(rag_result)

### 2.4. Recherche web

`web_search` lance une recherche web via la Tavily Search API et retourne titres, liens et extraits. `web_extract` récupère le contenu complet d'une ou plusieurs URL via la Tavily Extract API. Fournis dans `shared/agent_tools.py`.

In [ ]:
# Test : recherche web
search_result = web_search("meilleures périodes pour visiter Rome")
print(search_result)

---
## 3. Prompt système de l'agent

### 3.1. Rédiger le prompt système de base

On reprend le prompt système classique défini lors des TP précédents.

Cependant, l'agent peut avoir besoin d'**instructions spécifiques par outil** pour savoir comment les utiliser correctement.

Conseil : demandez à l'agent de **sourcer chaque fait** avec le nom de l'outil ou du document utilisé (ex: `[tool_get_weather : source]`).

In [ ]:
base_system_prompt = """

# RÈGLES

Tu es un assistant de planification de voyage.

## Règles générales

### Usage des outils
- Utiliser les outils avant de répondre dès qu'un fait est nécessaire.
- Chaîner les outils si besoin (date → géoloc → météo → docs → réponse).
- Ne jamais inventer de fait : tout doit provenir d'un résultat d'outil.
- Citer les preuves factuelles (ex: [tool_name : source]).
- Si une information manque, la lister clairement.

### Style de réponse
- Être concis et pratique.
- Pas de mise en forme Markdown décorative.
- Pour chaque recommandation : 1 raison courte + 1 détail pratique concret.

### Format attendu
1) Résumé
2) Recommandations détaillées
3) Informations manquantes

## Consignes spécifiques par outil

### tool_get_current_date
- Toujours l'appeler en premier si la requête mentionne une date relative ("la semaine prochaine", "dans 3 jours").

### tool_geocode_location
- Convertir un lieu en coordonnées avant d'appeler tool_get_weather ou tool_search_nearby.

### tool_get_weather
- Vérifier les conditions météo sur la période demandée avant de proposer des activités en extérieur.
- Ne couvre que le court terme : pour un horizon de plusieurs mois, s'appuyer plutôt sur web_search.

### tool_search_nearby
- Utiliser pour recommander des lieux concrets autour de coordonnées précises.

### tool_retrieve_docs
- Toujours consulter les guides de voyage internes avant de répondre sur les lieux à visiter.

### web_search / web_extract
- Utiliser pour des informations récentes ou sur un horizon long (tendances saisonnières, prix).

"""

---
## 4. Cas d'usage 1 : Rome en 4 jours

Par exemple, on peut fournir à l'agent :
- L'outil RAG pour récupérer des passages pertinents des guides de voyage
- L'outil météo pour vérifier la météo prévue sur les dates du voyage
- L'outil géocodage pour convertir les lieux en coordonnées GPS
- etc.

--> Veillez à ce que les **instructions soient généralistes**, et non pas spécifiquement adaptées à la requête utilisateur.

### 4.1. Configurer et lancer l'agent

In [ ]:
request_use_case_1 = (
    "Je vais à Rome la semaine prochaine pour 4 jours (du jeudi au dimanche), arrivée le matin, départ le soir. "
    "Fais un plan de 4 jours avec un budget de 300 EUR pour les sorties et restaurants. "
    "Évite les zones trop touristiques et privilégie les lieux confidentiels."
)

tools_use_case_1 = [tool_get_current_date, tool_geocode_location, tool_get_weather, tool_retrieve_docs]

agent_use_case_1 = LoggingAgent(
    model=model,
    instructions=base_system_prompt,
    model_settings=agent_model_settings,
    tools=tools_use_case_1,
)

result_use_case_1 = await agent_use_case_1.run_with_logging(
    request=request_use_case_1,
    log_path=LOG_DIR / "trace_use_case_1.log",
    max_steps=12,
)

### 4.2. Afficher la réponse

In [ ]:
print(result_use_case_1.output)

--> Consultez aussi le fichier de log (`TP3_travel_planner_Agent/logs/trace_use_case_1.log`) : il contient la trace complète des outils appelés, dans quel ordre, avec quels arguments et résultats, ce qui permet de voir ce que l'agent en a déduit à chaque étape.

---
## 5. Cas d'usage 2 : Meilleure période Paris → New York

### 5.1. Configurer et lancer l'agent

In [ ]:
request_use_case_2 = (
    "Trouve la meilleure période dans les 6 prochains mois pour un voyage Paris → New York. "
    "Compare météo et prix saisonniers, et justifie la recommandation. "
    "Précise les prix, le temps de trajet et les conditions météo."
)

tools_use_case_2 = [tool_get_current_date, tool_get_weather, web_search, web_extract]

agent_use_case_2 = LoggingAgent(
    model=model,
    instructions=base_system_prompt,
    model_settings=agent_model_settings,
    tools=tools_use_case_2,
)

result_use_case_2 = await agent_use_case_2.run_with_logging(
    request=request_use_case_2,
    log_path=LOG_DIR / "trace_use_case_2.log",
    max_steps=12,
)

### 5.2. Afficher la réponse

In [ ]:
print(result_use_case_2.output)

---
## 6. Cas d'usage 3 : Recommandations près d'une adresse

### 6.1. Configurer et lancer l'agent

In [ ]:
request_use_case_3 = (
    "Recommande les meilleurs restaurants et activités près de cette adresse : "
    "10 Rue de la Paix, 75002 Paris, France. "
    "Budget : 30 euros pour un repas, 20 euros pour une activité."
)

tools_use_case_3 = [tool_geocode_location, tool_search_nearby]

agent_use_case_3 = LoggingAgent(
    model=model,
    instructions=base_system_prompt,
    model_settings=agent_model_settings,
    tools=tools_use_case_3,
)

result_use_case_3 = await agent_use_case_3.run_with_logging(
    request=request_use_case_3,
    log_path=LOG_DIR / "trace_use_case_3.log",
    max_steps=12,
)

### 6.2. Afficher la réponse

In [ ]:
print(result_use_case_3.output)